# EV Battery Failure Prediction & Powertrain Diagnostics Pipeline
**Role:** Lead Machine Learning Engineer & Powertrain Diagnostics Specialist  
**Dataset:** `ev_battery_health.csv` (20,000 packs, 68 raw columns, 8 domain-physics engineered features, 4 novel degradation indices)  
**Target:** `battery_failure` (0 = Healthy, 1 = Failure, ~6.92% base failure rate)

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)
from xgboost import XGBClassifier

from feature_engine import clip_sensor_spikes, engineer_physics_features

print('Environment and modules loaded successfully!')

## 1. Data Ingestion & Sensor Quality Audit
BMS telemetry data often contains transient sensor spikes or anomalous ADC readings.
We clip raw temperatures to `[-30°C, 80°C]` and State of Health to `[0%, 100%]`.

In [2]:
df_raw = pd.read_csv('ev_battery_health.csv')
print(f'Telemetry records: {len(df_raw):,}, Attributes: {df_raw.shape[1]}')
print('Class balance:')
print(df_raw['battery_failure'].value_counts())

df_clean = clip_sensor_spikes(df_raw)
print('Sensor spikes clipped to physical bounds.')

## 2. Domain-Physics Feature Transformation
We compute 8 electrochemistry-informed features:
1. **Arrhenius Aging Factor**: Quantifies exponential SEI growth rate as a function of peak temperature.
2. **Cell Temperature Variance Delta**: Measures spatial thermal gradients across battery modules.
3. **Cell Voltage Spread**: Captures cell-to-cell capacity and impedance mismatch.
4. **Fast-Charge Cumulative Stress Index**: Tracks micro-crack propagation from high C-rate cycling.
5. **Resistance Growth Proxy**: Interacts Joule heating internal resistance with capacity loss.
6. **Equivalent Full Cycles (EFC) per 1,000 km**: Fleet utilization intensity metric.
7. **Composite Physical Stress Aggregate**: Joint thermal, charging, and driving stress sum.
8. **SoH Wear Loss per Cycle**: Rate of health degradation per charge/discharge cycle.

Plus 4 novel electrochemical metrics:
9. **Thermal Dissipation Strain**
10. **IR per kWh Nominal Capacity**
11. **Lithium Plating Susceptibility Metric**
12. **Degradation Acceleration Index**

In [3]:
df_feat = engineer_physics_features(df_clean)
physics_cols = [
    'arrhenius_aging_factor', 'cell_temperature_variance_delta', 'cell_voltage_spread',
    'fast_charge_cumulative_stress', 'resistance_growth_proxy', 'efc_per_1000km',
    'composite_physical_stress', 'soh_wear_loss_per_cycle',
    'thermal_dissipation_strain', 'ir_per_kwh', 'lithium_plating_risk', 'degradation_acceleration_index'
]
df_feat[physics_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]

## 3. Train/Test Stratified Split & Preprocessing Pipeline
We impute numeric missing values with median, encode categorical attributes (`battery_chemistry`, `drive_type`, `terrain_type`, etc.), and scale features.

In [4]:
from eda_and_model import run_pipeline

# Run the complete production training pipeline and serialize artifacts
bundle = run_pipeline('ev_battery_health.csv')
print('Pipeline executed and artifacts saved successfully!')

## 4. Evaluation Visualizations & Model Artifacts
Inspecting the Confusion Matrix, ROC Curve (AUC = 0.980), and Feature Importance weights.

In [5]:
from IPython.display import Image, display
if os.path.exists('model_evaluation_metrics.png'):
    display(Image(filename='model_evaluation_metrics.png'))
if os.path.exists('feature_importance.png'):
    display(Image(filename='feature_importance.png'))